In [1]:
## Restructured model with germany data

In [2]:
import pickle
with open('../datasets/data/polymod_germany.pkl', 'rb') as file:
    data = pickle.load(file)

In [3]:
from cntmosaic.dataloader import DataFrameLoader, GeneralLoader

In [4]:
import pandas as pd

In [5]:
d = GeneralLoader(data['contacts'])

In [6]:
d.data

,id,age_part,sex_part,occ_part,edu_part,hh_size,class_size,work_contacts_nr,age_cnt,sex_cnt,y
0,846,10.0,F,5.0,2.0,4,26.0,NaN,43.0,M,3.0
1,846,10.0,F,5.0,2.0,4,26.0,NaN,70.0,F,2.0
2,846,10.0,F,5.0,2.0,4,26.0,NaN,68.0,M,2.0
3,846,10.0,F,5.0,2.0,4,26.0,NaN,11.0,F,1.0
4,846,10.0,F,5.0,2.0,4,26.0,NaN,13.0,F,2.0
...,...,...,...,...,...,...,...,...,...,...,...
10648,2185,70.0,M,2.0,8.0,2,NaN,NaN,57.0,M,1.0
10649,2185,70.0,M,2.0,8.0,2,NaN,NaN,57.0,M,1.0
10650,2185,70.0,M,2.0,8.0,2,NaN,NaN,61.0,M,1.0
10651,2185,70.0,M,2.0,8.0,2,NaN,NaN,32.0,M,1.0


In [35]:
out = d.generate_input({'id_var':'id', 'grp_vars':['sex_part', 'sex_cnt']})

/home/yiminglin/Downloads/cntmosaic/cntmosaic/dataloader/loaders.py:57: RuntimeWarning: Dropped 340 rows with missing values
  warnings.warn(f'Dropped {n_dropped} rows with missing values', RuntimeWarning)


In [44]:
out

,age_part,age_cnt,sex_part,sex_cnt,y,N
0,0,0,F,M,0,5
1,0,0,F,F,0,5
2,0,0,M,M,0,8
3,0,0,M,F,0,6
4,0,1,F,M,0,5
...,...,...,...,...,...,...
27195,83,82,F,F,1,2
27196,83,83,F,M,0,2
27197,83,83,F,F,0,2
27198,83,84,F,M,0,2


In [8]:
help(d.generate_input)

Help on method generate_input in module cntmosaic.dataloader.loaders:

generate_input(column_map) method of cntmosaic.dataloader.loaders.GeneralLoader instance
    Argument:
    column_map, dict
    required key:
        y, contact counts
        id_var, unique identifier
        age_part, age for the participants
        age_cnt/age_grp_cnt, age for the contacts



In [9]:
from cntmosaic.models.restr_BRCfine import restr_BRCfine

In [36]:
model = restr_BRCfine(out)

new model instantiated, please check default hyperparameters


In [37]:
model.compile()

model compiled, ready for sampling


In [38]:
import jax
model.run_inference_mcmc(jax.random.PRNGKey(0),
    num_samples = 50,
    num_warmup = 50,
    num_chains = 2)

/home/yiminglin/Downloads/cntmosaic/cntmosaic/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 2 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(2)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 100/100 [07:04<00:00,  4.24s/it, 511 steps of size 1.27e-02. acc. prob=0.91]

Number of divergences: 1


In [14]:
from cntmosaic.sim import ModelEvaluatorMCMC

In [39]:
e = ModelEvaluatorMCMC(model)

In [51]:
e.post = model.mcmc.get_samples(group_by_chain=True)
#e.get_posterior() extra dimension for different chains

In [17]:
import numpy as np

In [ ]:
log_rate = e.post['log_rate']
post_cint = {}
for name, site in e.post.items():
    if 'log_delta' in name:
        var = name.split('/')[0]
        cat = e.model.data[var].cat.categories
        post_cint[var] = {
            cat[i]: np.exp(log_rate[:, None, :, :] + site + e.model._precompute.log_P[None, None, :, :])[:, i, :, :]
            for i in range(len(cat))
        }

In [45]:
e.post.keys()

dict_keys(['baseline', 'beta', 'beta0', 'inv_disp', 'log_cint', 'log_rate', 'rho'])

In [52]:
e.post['log_rate'].shape

(2, 50, 85, 85)